# Checkpoint-198 · Colab Pro · baseline / vLLM / RAG

Upload source ZIP, checkpoint-198-inference.zip, dataset/ids và đúng base đã train vào Drive. Sửa BASE_SOURCE_DIR/IDS_FILE/TRAIN_ROOT theo nơi đã lưu. Chọn GPU và đủ RAM hệ thống cho bước merge CPU trên Colab. Cài dependency chung một lần; merge một lần, cả ba lượt dùng cùng model merged. Checkpoint ghi stage ~0,5 epoch và parent adapter hash; tổng1,5 epoch là thông tin chủ dự án. Không load/thử full model trên laptop. Xem docs/colab-checkpoint198.md.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, stat, subprocess, runpy, time, hashlib

PLATFORM = "colab"  # colab | kaggle | gpu
MODEL_KEY = "qwen35-9b"
BACKEND = None  # None: config/env; "torch" or "vllm" to override
RAG_MODE = None  # None: config/env; "none", "dense", "dense_rerank"
OWNER = "Bảo"  # thinh or your name
RUN_NAME = time.strftime("%Y%m%d-%H%M%S")
RUN_LIMIT = None
IDS_FILE = None  # JSON array of benchmark question IDs
RESUME = False
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE = Path("/content/drive/MyDrive/DataMate")
    WORK = Path("/content/datamate")
    SOURCE_ZIP = STORAGE / "datamate-colab-source.zip"
    TRAIN_ROOT = STORAGE / "data/train"
    TRAIN_ZIP = STORAGE / "train.zip"
    MODEL_SOURCE_DIR = STORAGE / "models" / MODEL_KEY
elif PLATFORM == "kaggle":
    WORK = Path("/kaggle/working/datamate")
    STORAGE = Path("/kaggle/working/datamate-results")
    SOURCE_ZIP = Path("/kaggle/input/datamate-source/datamate-colab-source.zip")
    TRAIN_ROOT = Path("/kaggle/input/datamate-data/train")
    TRAIN_ZIP = None
    MODEL_SOURCE_DIR = Path("/kaggle/input/datamate-model") / MODEL_KEY
elif PLATFORM == "gpu":
    WORK = Path("/work/datamate")
    STORAGE = Path("/work/datamate-results")
    SOURCE_ZIP = Path("/work/inputs/datamate-colab-source.zip")
    TRAIN_ROOT = Path("/work/data/train")
    TRAIN_ZIP = None
    MODEL_SOURCE_DIR = Path("/work/models") / MODEL_KEY
else:
    raise ValueError("PLATFORM must be colab, kaggle or gpu")
WORK = Path(os.environ.get("DATAMATE_WORK_DIR", str(WORK)))
SOURCE_ZIP = Path(os.environ.get("DATAMATE_SOURCE_ZIP", str(SOURCE_ZIP)))
TRAIN_ROOT = Path(os.environ.get("DATAMATE_SPLIT_ROOT", str(TRAIN_ROOT)))
MODEL_SOURCE_DIR = Path(os.environ.get("DATAMATE_MODEL_DIR", str(MODEL_SOURCE_DIR)))
EMBEDDING_SOURCE_DIR = Path(os.environ.get("DATAMATE_RAG_EMBEDDING_DIR", str(MODEL_SOURCE_DIR.parent / "vietnamese-embedding")))
RERANKER_SOURCE_DIR = Path(os.environ.get("DATAMATE_RAG_RERANKER_DIR", str(MODEL_SOURCE_DIR.parent / "vietnamese-reranker")))
COPY_MODEL_TO_SCRATCH = PLATFORM == "colab"
SAVE = Path(os.environ.get("DATAMATE_RESULT_DIR", str(STORAGE / "runs" / OWNER / RUN_NAME)))
SAVE.mkdir(parents=True, exist_ok=True)
if SOURCE_ZIP.is_file():
    with SOURCE_ZIP.open("rb") as handle:
        source_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    WORK = WORK / ("source-" + source_hash[:12])
    receipt = WORK / ".source_archive_sha256"
    if WORK.exists() and any(WORK.iterdir()) and (not receipt.is_file() or receipt.read_text().strip() != source_hash):
        raise RuntimeError("Source workspace differs from ZIP; choose a new WORK folder")
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for entry in archive.infolist():
            target = (WORK / entry.filename).resolve()
            if not target.is_relative_to(WORK.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("Unsafe source ZIP path")
        archive.extractall(WORK)
    receipt.write_text(source_hash + "\n", encoding="utf-8")
elif not (WORK / "src/datamate/__init__.py").is_file():
    raise FileNotFoundError("Provide SOURCE_ZIP or an existing repository at WORK")
loaded = sys.modules.get("datamate")
if loaded and not Path(loaded.__file__).resolve().is_relative_to(WORK / "src"):
    raise RuntimeError("Another DataMate source is loaded; restart the kernel")
CONFIG_FILE = Path(os.environ["DATAMATE_CONFIG"]) if os.environ.get("DATAMATE_CONFIG") else WORK / "configs" / (MODEL_KEY + ".json")
if not TRAIN_ROOT.is_dir():
    if TRAIN_ZIP is None or not TRAIN_ZIP.is_file():
        raise FileNotFoundError("Set TRAIN_ROOT to an extracted split containing questions.jsonl")
    helpers = runpy.run_path(str(WORK / "tools/colab_archives.py"))
    TRAIN_ROOT = helpers["prepare_train_archive"](TRAIN_ZIP, WORK, WORK / "configs/training_sources.json")
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ.update(HF_HOME=str(WORK / ".cache/huggingface"), TORCH_HOME=str(WORK / ".cache/torch"),
                  PIP_CACHE_DIR=str(WORK / ".cache/pip"))
print("Source:", WORK, "| data:", TRAIN_ROOT, "| results:", SAVE)

# Upload ZIP source as datamate-colab-source.zip and the checkpoint ZIP below.
CHECKPOINT_ZIP = STORAGE / "checkpoint-198-inference.zip"
BASE_SOURCE_DIR = STORAGE / "models/qwen35-9b"  # Exact sealed base used for training.
MERGED_STORE = STORAGE / "models/qwen35-9b-checkpoint-198-merged"
EXPECTED_ADAPTER_SHA256 = "f346ad51cd37abe3ea5f9d127bd506f4c45f1341c836c8fbd54c86c6426a4ebe"
IDS_FILE = STORAGE / "benchmark/ids.json"  # Same frozen set for all three arms.
REFERENCES_FILE = None  # Optional gold JSONL; read only after inference.
COMPARE_OUTPUT = SAVE / "comparison"  # Keep fresh; preparation logs stay outside.
DTYPE = "float16"  # Common generator precision for all three arms.


## Cài stack chung trước baseline
Nếu đã import thư viện rồi pip đổi phiên bản, restart kernel và chạy lại.


In [ ]:
from importlib.metadata import version,PackageNotFoundError
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
loaded = {n:installed(n) for n in ("torch","transformers","tokenizers","numpy","peft","vllm") if n in sys.modules}
subprocess.run([sys.executable,"-m","pip","install","-e",".[llm,vllm,rag]","peft==0.21.2","psutil>=5"],cwd=WORK,check=True)
changed = [n for n,before in loaded.items() if installed(n)!=before]
if changed:
    raise RuntimeError("Restart kernel before inference; loaded packages changed: " + str(changed))
subprocess.run([sys.executable,"-m","pip","check"],check=True)
subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv"],check=True)
sys.path.insert(0,str(WORK / "src"))
print("Common stack installed. Model merge runs in a fresh CPU child; inference in fresh GPU workers.")


## Chuẩn bị checkpoint và model
Không train thêm. CPU merge cần RAM còn trống theo preflight; xem prepare-merge.log trên Drive nếu lỗi.


In [ ]:
from tools.checkpoint_inference import extract_package,verify_package
from tools.colab_model_cache import prepare_local_model
from datamate.models.artifacts import verify_artifact,sha256

package_root = extract_package(CHECKPOINT_ZIP,WORK / ".cache/checkpoint-198")
manifest = verify_package(package_root)
if manifest["files"]["adapter/adapter_model.safetensors"] != EXPECTED_ADAPTER_SHA256:
    raise RuntimeError("Uploaded checkpoint differs from the selected checkpoint-198")
print("Verified checkpoint:",manifest["source_global_step"],"stage epoch:",manifest["source_stage_epoch"])
base = BASE_SOURCE_DIR
if not (MERGED_STORE / "artifact_manifest.json").is_file():
    if not (base / "artifact_manifest.json").is_file() or sha256(base / "artifact_manifest.json") != manifest["expected_base_manifest_sha256"]:
        raise RuntimeError("BASE_SOURCE_DIR must point to the exact sealed training base (manifest " + manifest["expected_base_manifest_sha256"] + "); do not substitute another base")
    base = prepare_local_model(base,WORK / ".cache/models","qwen35-9b")
print("Merging/reusing checkpoint-198 on Colab; output:",MERGED_STORE,flush=True)
started = time.monotonic()
with (SAVE / "prepare-merge.log").open("w",encoding="utf-8") as log:
    subprocess.run([sys.executable,"-u",str(WORK / "tools/checkpoint_inference.py"),"merge",
        "--package-root",str(package_root),"--base",str(base),"--output",str(MERGED_STORE)],
        cwd=WORK,check=True,stdout=log,stderr=subprocess.STDOUT)
verify_artifact(MERGED_STORE,"qwen35-9b")
MODEL_DIR = prepare_local_model(MERGED_STORE,WORK / ".cache/models","qwen35-9b")
paths = {}
for key in ("vietnamese-embedding","vietnamese-reranker"):
    source = STORAGE / "models" / key
    if not (source / "artifact_manifest.json").is_file():
        # Explicit ONLINE preparation only, never inside benchmark inference.
        env = {**os.environ,"HF_HUB_OFFLINE":"0","TRANSFORMERS_OFFLINE":"0"}
        subprocess.run([sys.executable,"-m","datamate","stage-model","--model",key,"--output",str(source)],
                       cwd=WORK,env=env,check=True)
    verify_artifact(source,key)
    paths[key] = prepare_local_model(source,WORK / ".cache/models",key)
EMBEDDING_DIR,RERANKER_DIR = paths["vietnamese-embedding"],paths["vietnamese-reranker"]
(SAVE / "preparation.json").write_text(json.dumps({"seconds":time.monotonic()-started,
    "merged_model_dir":str(MODEL_DIR),"checkpoint_adapter_sha256":EXPECTED_ADAPTER_SHA256,
    "note":"Online staging/copy/one-time merge outside the three-arm inference timers"},indent=2))
print("All arms will use this merged artifact:",MODEL_DIR)


## Chạy baseline trước rồi vLLM và RAG
Cùng source/IDs/checkpoint/dtype/budget. Gold chỉ được dùng sau inference. Dùng thư mục comparison mới cho mỗi phép so.


In [ ]:
command = [sys.executable,"-u",str(WORK / "tools/compare_system.py"),
    "--config",str(WORK / "configs/comparison/baseline.json"),"--ids",str(IDS_FILE),
    "--split-root",str(TRAIN_ROOT),"--model-dir",str(MODEL_DIR),
    "--embedding-dir",str(EMBEDDING_DIR),"--reranker-dir",str(RERANKER_DIR),
    "--dtype",DTYPE,"--output",str(COMPARE_OUTPUT)]
if REFERENCES_FILE is not None:
    command += ["--references",str(REFERENCES_FILE)]
subprocess.run(command,cwd=WORK,check=True)
print("Results:",COMPARE_OUTPUT / "comparisons.json")
